In [1]:
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch
print("cuda available:", torch.cuda.is_available())
print("imported torch")
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
print("imported tokeniser")
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from datasets import Dataset
print("cuda" if torch.cuda.is_available() else "cpu")

# !git clone https://github.com/Tisya-V/NLP-CW.git
import sys
sys.path.insert(0, "/content/NLP-CW/src")

import utils

!nvidia-smi

cuda available: True
imported torch
imported tokeniser
cuda
Tue Mar  3 19:02:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |            

In [2]:
RANDOM_SEED = 47

train_df, dev_df, test_df = utils.load_and_clean_data()

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["binary_label"],  # ensures same class ratio in both splits
    random_state=RANDOM_SEED
)
!pip install nlpaug nltk
import nlpaug.augmenter.word as naw
import nltk

# Synonym replacement: replace up to 15% of words, max 3 swaps per sentence
sr_aug = naw.SynonymAug(
    aug_src='wordnet',
    aug_p=0.15,
    aug_max=3
)

def augment_pcl_rows(df, augmenter, n_copies=1, seed=RANDOM_SEED):
    pcl_rows = df[df["binary_label"] == 1].copy()
    augmented = []
    for _, row in pcl_rows.iterrows():
        for _ in range(n_copies):
            try:
                new_text = augmenter.augment(row["text"])[0]
            except Exception:
                new_text = row["text"]  # fallback: keep original if augmentation fails
            augmented.append({"text": new_text, "binary_label": 1})
    return pd.DataFrame(augmented)

aug_df = augment_pcl_rows(train_df, sr_aug, n_copies=3)
train_df_aug = pd.concat([train_df, aug_df], ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f"Original: {train_df['binary_label'].value_counts().to_dict()}")
print(f"Augmented: {train_df_aug['binary_label'].value_counts().to_dict()}")

# train_df = train_df_aug

# majority = train_df[train_df["binary_label"] == 0]
# minority = train_df[train_df["binary_label"] == 1]

# # Upsample minority to match majority
# minority_upsampled = resample(
#     minority,
#     replace=True,                    # sample with replacement
#     n_samples=len(majority) // 2,         # match majority class size
#     random_state=RANDOM_SEED
# )

# train_balanced = pd.concat([majority, minority_upsampled]).sample(
#     frac=1, random_state=RANDOM_SEED
# ).reset_index(drop=True)

# print(f"Original: {len(train_df)} | Balanced: {len(train_balanced)}")
# print(train_balanced["binary_label"].value_counts())

Data loaded successfully
Train: 8375 | Dev: 2094 | Test: 3831
train: 0 NaN | 0 empty | 1 <3 words
Short rows (<3 words):      par_id      text  binary_label
1620   1657  refugees             0
dev: 1 NaN | 0 empty | 1 <3 words
Short rows (<3 words):     par_id              text  binary_label
794   9064  Feeling hopeless             0
  Dropping 1 rows:
    par_id text
401   8640  NaN
test: 0 NaN | 0 empty | 0 <3 words


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

Original: {0: 6443, 1: 675}
Augmented: {0: 6443, 1: 2700}


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger t

In [3]:
MODEL_NAME = "roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(df):
    enc = tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    if "binary_label" in df.columns:
        enc["labels"] = df["binary_label"].tolist()
    return Dataset.from_dict(enc)

train_dataset = tokenize(train_df_aug)
val_dataset   = tokenize(val_df)
dev_dataset   = tokenize(dev_df)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [4]:
labels_array = train_df["binary_label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=labels_array
)
print(f"Class weights → 0: {class_weights[0]:.3f}, 1: {class_weights[1]:.3f}")

# Move to device for use in loss
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
device = torch.device(device)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

Class weights → 0: 0.552, 1: 5.273
Device: cuda


In [5]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.dtype))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss
    
from transformers import TrainerCallback

class PrinterCallback(TrainerCallback):
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}", flush=True)


In [6]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # F1 of positive class only — matches the task metric
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_pcl": f1}



In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir                  = "./model",
    num_train_epochs            = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_pcl",
    bf16                        = False,
    fp16                        = False,
    logging_steps               = 1,
    # report_to                 = "none",
    # disable_tqdm              = True,
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [PrinterCallback()],
)

trainer.train()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Pcl
1,0.170586,0.487994,0.447699
2,0.080237,0.620315,0.544892
3,0.136316,1.159138,0.560669
4,0.226372,1.164977,0.571429
5,0.001330,1.358114,0.564885


Step 1: {'loss': 0.6941704750061035, 'grad_norm': 3.984924793243408, 'learning_rate': 0.0, 'epoch': 0.0034965034965034965}
Step 2: {'loss': 0.7160443067550659, 'grad_norm': 3.9734456539154053, 'learning_rate': 6.993006993006993e-08, 'epoch': 0.006993006993006993}
Step 3: {'loss': 0.7114578485488892, 'grad_norm': 3.3464291095733643, 'learning_rate': 1.3986013986013987e-07, 'epoch': 0.01048951048951049}
Step 4: {'loss': 0.7327481508255005, 'grad_norm': 3.6312296390533447, 'learning_rate': 2.097902097902098e-07, 'epoch': 0.013986013986013986}
Step 5: {'loss': 0.6948844790458679, 'grad_norm': 3.791883945465088, 'learning_rate': 2.7972027972027973e-07, 'epoch': 0.017482517482517484}
Step 6: {'loss': 0.7315171957015991, 'grad_norm': 4.980594158172607, 'learning_rate': 3.496503496503497e-07, 'epoch': 0.02097902097902098}
Step 7: {'loss': 0.7049873471260071, 'grad_norm': 4.454082489013672, 'learning_rate': 4.195804195804196e-07, 'epoch': 0.024475524475524476}
Step 8: {'loss': 0.724318385124206

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Step 1145: {'loss': 0.0069016083143651485, 'grad_norm': 11.753021240234375, 'learning_rate': 2.222222222222222e-06, 'epoch': 4.003496503496503}
Step 1146: {'loss': 0.10692214220762253, 'grad_norm': 19.95701026916504, 'learning_rate': 2.2144522144522146e-06, 'epoch': 4.006993006993007}
Step 1147: {'loss': 0.029242942109704018, 'grad_norm': 8.951911926269531, 'learning_rate': 2.2066822066822067e-06, 'epoch': 4.010489510489511}
Step 1148: {'loss': 0.07037204504013062, 'grad_norm': 4.835004806518555, 'learning_rate': 2.1989121989121992e-06, 'epoch': 4.013986013986014}
Step 1149: {'loss': 0.007729135919362307, 'grad_norm': 6.71219539642334, 'learning_rate': 2.1911421911421913e-06, 'epoch': 4.0174825174825175}
Step 1150: {'loss': 0.0024800195824354887, 'grad_norm': 0.3788969814777374, 'learning_rate': 2.1833721833721834e-06, 'epoch': 4.020979020979021}
Step 1151: {'loss': 0.0020821839570999146, 'grad_norm': 0.19192923605442047, 'learning_rate': 2.175602175602176e-06, 'epoch': 4.0244755244755

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Step 1430: {'train_runtime': 2272.8794, 'train_samples_per_second': 20.113, 'train_steps_per_second': 0.629, 'total_flos': 6014060947891200.0, 'train_loss': 0.13986738380728536, 'epoch': 5.0}


TrainOutput(global_step=1430, training_loss=0.13986738380728536, metrics={'train_runtime': 2272.8794, 'train_samples_per_second': 20.113, 'train_steps_per_second': 0.629, 'total_flos': 6014060947891200.0, 'train_loss': 0.13986738380728536, 'epoch': 5.0})

In [11]:
from scipy.special import softmax

# threshhold tuning on val_df
val_output   = trainer.predict(val_dataset)
val_logits   = val_output.predictions
val_probs    = softmax(val_logits, axis=-1)[:, 1]   # P(PCL)
val_labels   = val_df["binary_label"].values

best_thresh, best_f1 = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 91):
    preds = (val_probs >= t).astype(int)
    f1    = f1_score(val_labels, preds, pos_label=1)
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print(f"Best threshold: {best_thresh:.2f}  |  Val F1: {best_f1:.4f}")

def get_predictions(trainer, dataset, threshold = 0.5):
    preds_output = trainer.predict(dataset)
    logits = preds_output.predictions
    probs = softmax(logits, axis=1)[:, 1]
    preds = (probs >= threshold).astype(int)
    return preds

# Dev predictions
dev_preds = get_predictions(trainer, dev_dataset, threshold = best_thresh)
print(classification_report(dev_df["binary_label"].values, dev_preds, target_names=["No PCL", "PCL"]))
np.savetxt("dev.txt", dev_preds.astype(int), fmt="%d")

# Test predictions (no labels)
test_dataset = tokenize(test_df)
test_preds = get_predictions(trainer, test_dataset, threshold = best_thresh)
np.savetxt("test.txt", test_preds.astype(int), fmt="%d")


Best threshold: 0.66  |  Val F1: 0.5809


              precision    recall  f1-score   support

      No PCL       0.97      0.94      0.95      1894
         PCL       0.53      0.69      0.60       199

    accuracy                           0.91      2093
   macro avg       0.75      0.81      0.77      2093
weighted avg       0.92      0.91      0.92      2093

